# 01 — Data Exploration: EGFR Drug–Target Evidence (ChEMBL)

First dataset for the Therapeutic Strategy Assistant.
Flow: **EGFR → ChEMBL target → activity records → drug-target dataset → CSV**.

*Resilient:* retries on ChEMBL hiccups, and falls back to the known EGFR id `CHEMBL203` if the search fails.

### 1. Test notebook environment

In [1]:
import sys
import time
from pathlib import Path

import requests
import pandas as pd

print("Notebook is working")
print("Python executable:", sys.executable)

Notebook is working
Python executable: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/.venv/bin/python


### 2. Helper: GET with retries
ChEMBL sometimes returns a temporary 500 or times out. This helper retries a few times so a hiccup does not crash the notebook.

In [2]:
def chembl_get(url, params, retries=4, pause=3):
    """GET JSON from ChEMBL, retrying on 500 / timeout. Returns dict or None."""
    for attempt in range(retries):
        try:
            response = requests.get(url, params=params, timeout=(10, 120))
            if response.status_code == 200:
                return response.json()
            print(f"  attempt {attempt + 1}: HTTP {response.status_code}, retrying...")
        except requests.exceptions.RequestException as error:
            print(f"  attempt {attempt + 1}: {type(error).__name__}, retrying...")
        time.sleep(pause)
    return None

### 3. Set project folders

In [3]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Processed data folder:", PROCESSED_DIR)

Project root: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant
Processed data folder: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/processed


### 4. Choose target

In [4]:
target_name = "EGFR"

print("Target selected:", target_name)

Target selected: EGFR


### 5. Search ChEMBL for the target (with fallback)

In [5]:
chembl_target_search_url = "https://www.ebi.ac.uk/chembl/api/data/target/search.json"

target_search_data = chembl_get(chembl_target_search_url, {"q": target_name})

if target_search_data is not None:
    print("ChEMBL target search worked.")
    print(target_search_data.keys())
else:
    print("ChEMBL target search failed after retries -> will use fallback CHEMBL203.")

ChEMBL target search worked.
dict_keys(['page_meta', 'targets'])


### 6. Convert target results to dataframe

In [6]:
if target_search_data is not None:
    targets_df = pd.DataFrame(target_search_data.get("targets", []))
    print("Number of targets found:", len(targets_df))
else:
    targets_df = pd.DataFrame()

useful_target_columns = ["target_chembl_id", "pref_name", "organism", "target_type"]
available_target_columns = [c for c in useful_target_columns if c in targets_df.columns] or useful_target_columns

if not targets_df.empty:
    display(targets_df[available_target_columns].head(20))

Number of targets found: 20


,target_chembl_id,pref_name,organism,target_type
0,CHEMBL4523747,EGFR/PPP1CA,Homo sapiens,PROTEIN-PROTEIN INTERACTION
1,CHEMBL5465557,CCN2-EGFR,Homo sapiens,PROTEIN-PROTEIN INTERACTION
2,CHEMBL3608,Epidermal growth factor receptor,Mus musculus,SINGLE PROTEIN
3,CHEMBL6193842,Protein cereblon/Epidermal growth factor receptor,Mus musculus,PROTEIN-PROTEIN INTERACTION
4,CHEMBL203,Epidermal growth factor receptor,Homo sapiens,SINGLE PROTEIN
5,CHEMBL4523680,Protein cereblon/Epidermal growth factor receptor,Homo sapiens,PROTEIN-PROTEIN INTERACTION
6,CHEMBL2363049,Epidermal growth factor receptor,Homo sapiens,PROTEIN FAMILY
7,CHEMBL3137284,MER intracellular domain/EGFR extracellular do...,Homo sapiens,CHIMERIC PROTEIN
8,CHEMBL4523998,von Hippel-Lindau disease tumor suppressor/Epi...,Homo sapiens,PROTEIN-PROTEIN INTERACTION
9,CHEMBL6193841,Protein cereblon/Epidermal growth factor receptor,Mus musculus,PROTEIN-PROTEIN INTERACTION


### 7. Keep human single-protein targets

In [7]:
if not targets_df.empty:
    human_targets = targets_df[
        (targets_df["organism"].str.contains("Homo sapiens", case=False, na=False))
        & (targets_df["target_type"] == "SINGLE PROTEIN")
    ].copy()
    display(human_targets[available_target_columns].head(10))
else:
    human_targets = pd.DataFrame()
    print("No targets dataframe; will use fallback.")

,target_chembl_id,pref_name,organism,target_type
4,CHEMBL203,Epidermal growth factor receptor,Homo sapiens,SINGLE PROTEIN
10,CHEMBL1824,Receptor tyrosine-protein kinase erbB-2,Homo sapiens,SINGLE PROTEIN
12,CHEMBL3009,Receptor tyrosine-protein kinase erbB-4,Homo sapiens,SINGLE PROTEIN
13,CHEMBL5838,Receptor tyrosine-protein kinase erbB-3,Homo sapiens,SINGLE PROTEIN


### 8. Pick the target id (prefer CHEMBL203)

In [8]:
if not human_targets.empty:
    exact = human_targets[human_targets["target_chembl_id"] == "CHEMBL203"]
    selected_target = exact.iloc[0] if not exact.empty else human_targets.iloc[0]
    target_chembl_id = selected_target["target_chembl_id"]
    target_pref_name = selected_target["pref_name"]
else:
    target_chembl_id = "CHEMBL203"
    target_pref_name = "Epidermal growth factor receptor"

print("Selected target ChEMBL ID:", target_chembl_id)
print("Selected target name:", target_pref_name)

Selected target ChEMBL ID: CHEMBL203
Selected target name: Epidermal growth factor receptor


### 9. Get ChEMBL activity records
We filter to records that have a `pchembl_value` (a potency score). This keeps the response light so ChEMBL does not return a 500, and gives us better-quality data.

In [9]:
chembl_activity_url = "https://www.ebi.ac.uk/chembl/api/data/activity.json"

activity_params = {
    "target_chembl_id": target_chembl_id,
    "pchembl_value__isnull": "false",
    "limit": 20,
}

activity_data = chembl_get(chembl_activity_url, activity_params)

if activity_data is None:
    raise RuntimeError("ChEMBL activity endpoint failed after retries. Try again shortly.")

print("Activity search worked.")
print(activity_data.keys())

Activity search worked.
dict_keys(['activities', 'page_meta'])


### 10. Convert activity records to dataframe

In [10]:
activities = activity_data.get("activities", [])
activities_df = pd.DataFrame(activities)
print("Number of activity records found:", len(activities_df))
activities_df.head()

Number of activity records found: 20


,action_type,activity_comment,activity_id,activity_properties,assay_chembl_id,assay_description,assay_type,assay_variant_accession,assay_variant_mutation,bao_endpoint,...,target_organism,target_pref_name,target_tax_id,text_value,toid,type,units,uo_units,upper_value,value
0,None,None,32260,[],CHEMBL674637,Inhibitory activity towards tyrosine phosphory...,B,None,None,BAO_0000190,...,Homo sapiens,Epidermal growth factor receptor,9606,None,None,IC50,uM,UO_0000065,None,0.041
1,None,None,32263,[],CHEMBL621151,Inhibition of autophosphorylation of human epi...,F,None,None,BAO_0000190,...,Homo sapiens,Epidermal growth factor receptor,9606,None,None,IC50,uM,UO_0000065,None,0.3
2,None,None,32265,[],CHEMBL615325,Inhibition of ligand-induced proliferation in ...,F,None,None,BAO_0000190,...,Homo sapiens,Epidermal growth factor receptor,9606,None,None,IC50,uM,UO_0000065,None,7.82
3,None,None,32267,[],CHEMBL674637,Inhibitory activity towards tyrosine phosphory...,B,None,None,BAO_0000190,...,Homo sapiens,Epidermal growth factor receptor,9606,None,None,IC50,uM,UO_0000065,None,0.17
4,None,None,32270,[],CHEMBL621151,Inhibition of autophosphorylation of human epi...,F,None,None,BAO_0000190,...,Homo sapiens,Epidermal growth factor receptor,9606,None,None,IC50,uM,UO_0000065,None,0.04


### 11. Keep useful activity columns

In [11]:
useful_activity_columns = [
    "molecule_chembl_id", "target_chembl_id", "target_organism", "target_pref_name",
    "assay_chembl_id", "assay_description", "standard_type", "standard_value",
    "standard_units", "pchembl_value", "document_chembl_id",
]

available_activity_columns = [c for c in useful_activity_columns if c in activities_df.columns]
clean_activities_df = activities_df[available_activity_columns].copy()
print("Rows:", len(clean_activities_df))
clean_activities_df.head(20)

Rows: 20


,molecule_chembl_id,target_chembl_id,target_organism,target_pref_name,assay_chembl_id,assay_description,standard_type,standard_value,standard_units,pchembl_value,document_chembl_id
0,CHEMBL68920,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL674637,Inhibitory activity towards tyrosine phosphory...,IC50,41.0,nM,7.39,CHEMBL1134862
1,CHEMBL68920,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL621151,Inhibition of autophosphorylation of human epi...,IC50,300.0,nM,6.52,CHEMBL1134862
2,CHEMBL68920,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL615325,Inhibition of ligand-induced proliferation in ...,IC50,7820.0,nM,5.11,CHEMBL1134862
3,CHEMBL69960,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL674637,Inhibitory activity towards tyrosine phosphory...,IC50,170.0,nM,6.77,CHEMBL1134862
4,CHEMBL69960,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL621151,Inhibition of autophosphorylation of human epi...,IC50,40.0,nM,7.40,CHEMBL1134862
5,CHEMBL69960,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL615325,Inhibition of ligand-induced proliferation in ...,IC50,440.0,nM,6.36,CHEMBL1134862
6,CHEMBL137635,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL677833,In vitro inhibition of Epidermal growth factor...,IC50,9300.0,nM,5.03,CHEMBL1145114
7,CHEMBL77085,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL674643,Inhibitory concentration of EGF dependent auto...,IC50,96000.0,nM,4.02,CHEMBL1124610
8,CHEMBL77085,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL675636,Inhibitory concentration of EGF dependent auto...,Ki,24000.0,nM,4.62,CHEMBL1124610
9,CHEMBL443268,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL674637,Inhibitory activity towards tyrosine phosphory...,IC50,5310.0,nM,5.28,CHEMBL1134862


### 12. Build the drug-target dataset

In [12]:
clean_activities_df = clean_activities_df.dropna(subset=["molecule_chembl_id"]).copy()

drug_target_activity_df = clean_activities_df.drop_duplicates(
    subset=["molecule_chembl_id", "target_chembl_id"]
).copy()

drug_target_activity_df["target_name"] = target_pref_name
drug_target_activity_df["relationship_source"] = "ChEMBL activity"

print("Unique molecule-target relationships:", len(drug_target_activity_df))
drug_target_activity_df.head(20)

Unique molecule-target relationships: 12


,molecule_chembl_id,target_chembl_id,target_organism,target_pref_name,assay_chembl_id,assay_description,standard_type,standard_value,standard_units,pchembl_value,document_chembl_id,target_name,relationship_source
0,CHEMBL68920,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL674637,Inhibitory activity towards tyrosine phosphory...,IC50,41.0,nM,7.39,CHEMBL1134862,Epidermal growth factor receptor,ChEMBL activity
3,CHEMBL69960,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL674637,Inhibitory activity towards tyrosine phosphory...,IC50,170.0,nM,6.77,CHEMBL1134862,Epidermal growth factor receptor,ChEMBL activity
6,CHEMBL137635,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL677833,In vitro inhibition of Epidermal growth factor...,IC50,9300.0,nM,5.03,CHEMBL1145114,Epidermal growth factor receptor,ChEMBL activity
7,CHEMBL77085,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL674643,Inhibitory concentration of EGF dependent auto...,IC50,96000.0,nM,4.02,CHEMBL1124610,Epidermal growth factor receptor,ChEMBL activity
9,CHEMBL443268,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL674637,Inhibitory activity towards tyrosine phosphory...,IC50,5310.0,nM,5.28,CHEMBL1134862,Epidermal growth factor receptor,ChEMBL activity
10,CHEMBL76589,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL674643,Inhibitory concentration of EGF dependent auto...,IC50,125.0,nM,6.90,CHEMBL1124610,Epidermal growth factor receptor,ChEMBL activity
11,CHEMBL76904,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL674643,Inhibitory concentration of EGF dependent auto...,IC50,35000.0,nM,4.46,CHEMBL1124610,Epidermal growth factor receptor,ChEMBL activity
13,CHEMBL304271,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL674637,Inhibitory activity towards tyrosine phosphory...,IC50,0.45,nM,9.35,CHEMBL1134862,Epidermal growth factor receptor,ChEMBL activity
14,CHEMBL296407,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL674643,Inhibitory concentration of EGF dependent auto...,IC50,10000.0,nM,5.00,CHEMBL1124610,Epidermal growth factor receptor,ChEMBL activity
16,CHEMBL309625,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL674643,Inhibitory concentration of EGF dependent auto...,IC50,60000.0,nM,4.22,CHEMBL1124610,Epidermal growth factor receptor,ChEMBL activity


### 13. Add a simple confidence score

In [13]:
def calculate_activity_confidence(row):
    score = 0.4
    if pd.notna(row.get("pchembl_value")):
        score += 0.25
    if pd.notna(row.get("standard_value")):
        score += 0.15
    if pd.notna(row.get("assay_description")):
        score += 0.1
    if pd.notna(row.get("document_chembl_id")):
        score += 0.1
    return round(min(score, 1.0), 2)


drug_target_activity_df["confidence_score"] = drug_target_activity_df.apply(
    calculate_activity_confidence, axis=1
)
drug_target_activity_df.head(20)

,molecule_chembl_id,target_chembl_id,target_organism,target_pref_name,assay_chembl_id,assay_description,standard_type,standard_value,standard_units,pchembl_value,document_chembl_id,target_name,relationship_source,confidence_score
0,CHEMBL68920,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL674637,Inhibitory activity towards tyrosine phosphory...,IC50,41.0,nM,7.39,CHEMBL1134862,Epidermal growth factor receptor,ChEMBL activity,1.0
3,CHEMBL69960,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL674637,Inhibitory activity towards tyrosine phosphory...,IC50,170.0,nM,6.77,CHEMBL1134862,Epidermal growth factor receptor,ChEMBL activity,1.0
6,CHEMBL137635,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL677833,In vitro inhibition of Epidermal growth factor...,IC50,9300.0,nM,5.03,CHEMBL1145114,Epidermal growth factor receptor,ChEMBL activity,1.0
7,CHEMBL77085,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL674643,Inhibitory concentration of EGF dependent auto...,IC50,96000.0,nM,4.02,CHEMBL1124610,Epidermal growth factor receptor,ChEMBL activity,1.0
9,CHEMBL443268,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL674637,Inhibitory activity towards tyrosine phosphory...,IC50,5310.0,nM,5.28,CHEMBL1134862,Epidermal growth factor receptor,ChEMBL activity,1.0
10,CHEMBL76589,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL674643,Inhibitory concentration of EGF dependent auto...,IC50,125.0,nM,6.90,CHEMBL1124610,Epidermal growth factor receptor,ChEMBL activity,1.0
11,CHEMBL76904,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL674643,Inhibitory concentration of EGF dependent auto...,IC50,35000.0,nM,4.46,CHEMBL1124610,Epidermal growth factor receptor,ChEMBL activity,1.0
13,CHEMBL304271,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL674637,Inhibitory activity towards tyrosine phosphory...,IC50,0.45,nM,9.35,CHEMBL1134862,Epidermal growth factor receptor,ChEMBL activity,1.0
14,CHEMBL296407,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL674643,Inhibitory concentration of EGF dependent auto...,IC50,10000.0,nM,5.00,CHEMBL1124610,Epidermal growth factor receptor,ChEMBL activity,1.0
16,CHEMBL309625,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL674643,Inhibitory concentration of EGF dependent auto...,IC50,60000.0,nM,4.22,CHEMBL1124610,Epidermal growth factor receptor,ChEMBL activity,1.0


### 14. Save the dataset

In [14]:
activity_file = PROCESSED_DIR / f"{target_name.lower()}_chembl_activity.csv"
drug_target_activity_df.to_csv(activity_file, index=False)
print("Saved:", activity_file)

Saved: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/processed/egfr_chembl_activity.csv


### 15. Check saved files

In [15]:
for file in PROCESSED_DIR.glob("*.csv"):
    print(file.name)

egfr_drug_recommendations.csv
egfr_chembl_mechanisms.csv
egfr_chembl_activity.csv


### 16. Final result

In [16]:
print(f"Therapeutic Strategy Assistant Dataset for Target: {target_name}")
print("=" * 70)
print(f"Selected target ID: {target_chembl_id}")
print(f"Selected target name: {target_pref_name}")
print(f"Activity records: {len(clean_activities_df)}")
print(f"Unique molecule-target relationships: {len(drug_target_activity_df)}")
drug_target_activity_df.head(20)

Therapeutic Strategy Assistant Dataset for Target: EGFR
Selected target ID: CHEMBL203
Selected target name: Epidermal growth factor receptor
Activity records: 20
Unique molecule-target relationships: 12


,molecule_chembl_id,target_chembl_id,target_organism,target_pref_name,assay_chembl_id,assay_description,standard_type,standard_value,standard_units,pchembl_value,document_chembl_id,target_name,relationship_source,confidence_score
0,CHEMBL68920,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL674637,Inhibitory activity towards tyrosine phosphory...,IC50,41.0,nM,7.39,CHEMBL1134862,Epidermal growth factor receptor,ChEMBL activity,1.0
3,CHEMBL69960,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL674637,Inhibitory activity towards tyrosine phosphory...,IC50,170.0,nM,6.77,CHEMBL1134862,Epidermal growth factor receptor,ChEMBL activity,1.0
6,CHEMBL137635,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL677833,In vitro inhibition of Epidermal growth factor...,IC50,9300.0,nM,5.03,CHEMBL1145114,Epidermal growth factor receptor,ChEMBL activity,1.0
7,CHEMBL77085,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL674643,Inhibitory concentration of EGF dependent auto...,IC50,96000.0,nM,4.02,CHEMBL1124610,Epidermal growth factor receptor,ChEMBL activity,1.0
9,CHEMBL443268,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL674637,Inhibitory activity towards tyrosine phosphory...,IC50,5310.0,nM,5.28,CHEMBL1134862,Epidermal growth factor receptor,ChEMBL activity,1.0
10,CHEMBL76589,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL674643,Inhibitory concentration of EGF dependent auto...,IC50,125.0,nM,6.90,CHEMBL1124610,Epidermal growth factor receptor,ChEMBL activity,1.0
11,CHEMBL76904,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL674643,Inhibitory concentration of EGF dependent auto...,IC50,35000.0,nM,4.46,CHEMBL1124610,Epidermal growth factor receptor,ChEMBL activity,1.0
13,CHEMBL304271,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL674637,Inhibitory activity towards tyrosine phosphory...,IC50,0.45,nM,9.35,CHEMBL1134862,Epidermal growth factor receptor,ChEMBL activity,1.0
14,CHEMBL296407,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL674643,Inhibitory concentration of EGF dependent auto...,IC50,10000.0,nM,5.00,CHEMBL1124610,Epidermal growth factor receptor,ChEMBL activity,1.0
16,CHEMBL309625,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL674643,Inhibitory concentration of EGF dependent auto...,IC50,60000.0,nM,4.22,CHEMBL1124610,Epidermal growth factor receptor,ChEMBL activity,1.0


### 17. Get EGFR drug mechanisms

In [17]:
chembl_mechanism_url = "https://www.ebi.ac.uk/chembl/api/data/mechanism.json"

mechanism_data = chembl_get(chembl_mechanism_url, {"target_chembl_id": target_chembl_id, "limit": 100})

if mechanism_data is None:
    raise RuntimeError("ChEMBL mechanism endpoint failed after retries. Try again shortly.")

mechanisms = mechanism_data.get("mechanisms", [])
mechanisms_df = pd.DataFrame(mechanisms)
print("Mechanism records found:", len(mechanisms_df))
mechanisms_df.head()

Mechanism records found: 90


,action_type,binding_site_comment,direct_interaction,disease_efficacy,max_phase,mec_id,mechanism_comment,mechanism_of_action,mechanism_refs,molecular_mechanism,molecule_chembl_id,parent_molecule_chembl_id,record_id,selectivity_comment,site_id,target_chembl_id,variant_sequence
0,INHIBITOR,None,1,1,4,241,NaN,Epidermal growth factor receptor erbB1 inhibitor,[{'ref_id': 'setid=e0fa4bca-f245-4d92-ae29-b0c...,1,CHEMBL1201827,CHEMBL1201827,1390843,NaN,None,CHEMBL203,None
1,INHIBITOR,None,1,1,4,242,NaN,Epidermal growth factor receptor erbB1 inhibitor,[{'ref_id': 'setid=8bc6397e-4bd8-4d37-a007-a32...,1,CHEMBL1201577,CHEMBL1201577,1344036,NaN,None,CHEMBL203,None
2,INHIBITOR,None,1,1,4,243,NaN,Epidermal growth factor receptor erbB1 inhibitor,[{'ref_id': 'setid=57bccb29-1c47-4c64-ab6a-779...,1,CHEMBL1079742,CHEMBL553,1344174,NaN,None,CHEMBL203,None
3,INHIBITOR,None,1,1,4,244,NaN,Epidermal growth factor receptor erbB1 inhibitor,[{'ref_id': 'setid=827d60e8-7e07-41b7-c28b-49e...,1,CHEMBL939,CHEMBL939,1344994,NaN,None,CHEMBL203,None
4,INHIBITOR,None,1,1,4,245,NaN,Epidermal growth factor receptor erbB1 inhibitor,[{'ref_id': 'setid=63319b01-cad6-4d0a-c39b-938...,1,CHEMBL1201179,CHEMBL554,1343218,NaN,None,CHEMBL203,None


### 18. Keep useful mechanism columns

In [18]:
useful_mechanism_columns = [
    "molecule_chembl_id", "mechanism_of_action", "action_type",
    "target_chembl_id", "mechanism_refs",
]
available_mechanism_columns = [c for c in useful_mechanism_columns if c in mechanisms_df.columns]
clean_mechanisms_df = mechanisms_df[available_mechanism_columns].drop_duplicates(subset=["molecule_chembl_id"]).copy()
print("Unique drugs with a mechanism:", len(clean_mechanisms_df))
clean_mechanisms_df.head(20)

Unique drugs with a mechanism: 76


,molecule_chembl_id,mechanism_of_action,action_type,target_chembl_id,mechanism_refs
0,CHEMBL1201827,Epidermal growth factor receptor erbB1 inhibitor,INHIBITOR,CHEMBL203,[{'ref_id': 'setid=e0fa4bca-f245-4d92-ae29-b0c...
1,CHEMBL1201577,Epidermal growth factor receptor erbB1 inhibitor,INHIBITOR,CHEMBL203,[{'ref_id': 'setid=8bc6397e-4bd8-4d37-a007-a32...
2,CHEMBL1079742,Epidermal growth factor receptor erbB1 inhibitor,INHIBITOR,CHEMBL203,[{'ref_id': 'setid=57bccb29-1c47-4c64-ab6a-779...
3,CHEMBL939,Epidermal growth factor receptor erbB1 inhibitor,INHIBITOR,CHEMBL203,[{'ref_id': 'setid=827d60e8-7e07-41b7-c28b-49e...
4,CHEMBL1201179,Epidermal growth factor receptor erbB1 inhibitor,INHIBITOR,CHEMBL203,[{'ref_id': 'setid=63319b01-cad6-4d0a-c39b-938...
5,CHEMBL2105712,Epidermal growth factor receptor erbB1 inhibitor,INHIBITOR,CHEMBL203,[{'ref_id': 'fd638e5e-8032-e7ca-0179-95e96ab5d...
6,CHEMBL3545063,Epidermal growth factor receptor erbB1 inhibitor,INHIBITOR,CHEMBL203,[{'ref_id': 'setid=5e81b4a7-b971-45e1-9c31-29c...
7,CHEMBL1743047,Epidermal growth factor receptor erbB1 inhibitor,INHIBITOR,CHEMBL203,[{'ref_id': 'setid=89bcf553-669a-40b0-a9d7-67a...
8,CHEMBL1645462,Epidermal growth factor receptor erbB1 inhibitor,INHIBITOR,CHEMBL203,"[{'ref_id': '17062696', 'ref_type': 'PubMed', ..."
9,CHEMBL1947204,Epidermal growth factor receptor erbB1 inhibitor,INHIBITOR,CHEMBL203,"[{'ref_id': '21789172', 'ref_type': 'PubMed', ..."


### 19. Helper: look up a drug name + approval phase
Each `molecule_chembl_id` (e.g. CHEMBL939) is turned into a readable name (e.g. GEFITINIB) via ChEMBL's molecule endpoint. `max_phase = 4` means the drug is **approved**.

In [19]:
def get_molecule_info(chembl_id):
    """Return (pref_name, max_phase) for a molecule id, or (None, None)."""
    url = f"https://www.ebi.ac.uk/chembl/api/data/molecule/{chembl_id}.json"
    data = chembl_get(url, params=None)
    if data is None:
        return None, None
    return data.get("pref_name"), data.get("max_phase")

### 20. Enrich each drug with its name (this calls the API per drug, so it takes ~1 min)

In [20]:
names, phases = [], []
for i, cid in enumerate(clean_mechanisms_df["molecule_chembl_id"], start=1):
    name, phase = get_molecule_info(cid)
    names.append(name)
    phases.append(phase)
    if i % 10 == 0:
        print(f"  enriched {i}/{len(clean_mechanisms_df)}")

clean_mechanisms_df["drug_name"] = names
clean_mechanisms_df["max_phase"] = phases
print("Done. Drugs with a real name:", clean_mechanisms_df["drug_name"].notna().sum())
clean_mechanisms_df.head(20)

  enriched 10/76
  enriched 20/76
  enriched 30/76
  enriched 40/76
  enriched 50/76
  enriched 60/76
  enriched 70/76
Done. Drugs with a real name: 76


,molecule_chembl_id,mechanism_of_action,action_type,target_chembl_id,mechanism_refs,drug_name,max_phase
0,CHEMBL1201827,Epidermal growth factor receptor erbB1 inhibitor,INHIBITOR,CHEMBL203,[{'ref_id': 'setid=e0fa4bca-f245-4d92-ae29-b0c...,PANITUMUMAB,4.0
1,CHEMBL1201577,Epidermal growth factor receptor erbB1 inhibitor,INHIBITOR,CHEMBL203,[{'ref_id': 'setid=8bc6397e-4bd8-4d37-a007-a32...,CETUXIMAB,4.0
2,CHEMBL1079742,Epidermal growth factor receptor erbB1 inhibitor,INHIBITOR,CHEMBL203,[{'ref_id': 'setid=57bccb29-1c47-4c64-ab6a-779...,ERLOTINIB HYDROCHLORIDE,4.0
3,CHEMBL939,Epidermal growth factor receptor erbB1 inhibitor,INHIBITOR,CHEMBL203,[{'ref_id': 'setid=827d60e8-7e07-41b7-c28b-49e...,GEFITINIB,4.0
4,CHEMBL1201179,Epidermal growth factor receptor erbB1 inhibitor,INHIBITOR,CHEMBL203,[{'ref_id': 'setid=63319b01-cad6-4d0a-c39b-938...,LAPATINIB DITOSYLATE,4.0
5,CHEMBL2105712,Epidermal growth factor receptor erbB1 inhibitor,INHIBITOR,CHEMBL203,[{'ref_id': 'fd638e5e-8032-e7ca-0179-95e96ab5d...,AFATINIB DIMALEATE,4.0
6,CHEMBL3545063,Epidermal growth factor receptor erbB1 inhibitor,INHIBITOR,CHEMBL203,[{'ref_id': 'setid=5e81b4a7-b971-45e1-9c31-29c...,OSIMERTINIB MESYLATE,4.0
7,CHEMBL1743047,Epidermal growth factor receptor erbB1 inhibitor,INHIBITOR,CHEMBL203,[{'ref_id': 'setid=89bcf553-669a-40b0-a9d7-67a...,NECITUMUMAB,4.0
8,CHEMBL1645462,Epidermal growth factor receptor erbB1 inhibitor,INHIBITOR,CHEMBL203,"[{'ref_id': '17062696', 'ref_type': 'PubMed', ...",AC-480,1.0
9,CHEMBL1947204,Epidermal growth factor receptor erbB1 inhibitor,INHIBITOR,CHEMBL203,"[{'ref_id': '21789172', 'ref_type': 'PubMed', ...",ALLITINIB,2.0


### 21. Build the drug recommendations table

In [21]:
drug_recommendations_df = clean_mechanisms_df[
    clean_mechanisms_df["drug_name"].notna()
].copy()

# approval label from max_phase
def approval_label(phase):
    if phase == 4:
        return "Approved"
    if phase in (1, 2, 3):
        return f"Investigational (Phase {int(phase)})"
    return "Research / Unknown"

drug_recommendations_df["approval_status"] = drug_recommendations_df["max_phase"].apply(approval_label)
drug_recommendations_df["target_name"] = target_pref_name

drug_recommendations_df = drug_recommendations_df[[
    "drug_name", "molecule_chembl_id", "action_type",
    "mechanism_of_action", "approval_status", "max_phase", "target_name",
]].sort_values("max_phase", ascending=False, na_position="last").reset_index(drop=True)

print("Drug recommendations:", len(drug_recommendations_df))
drug_recommendations_df.head(25)

Drug recommendations: 76


,drug_name,molecule_chembl_id,action_type,mechanism_of_action,approval_status,max_phase,target_name
0,PANITUMUMAB,CHEMBL1201827,INHIBITOR,Epidermal growth factor receptor erbB1 inhibitor,Research / Unknown,4.0,Epidermal growth factor receptor
1,CETUXIMAB,CHEMBL1201577,INHIBITOR,Epidermal growth factor receptor erbB1 inhibitor,Research / Unknown,4.0,Epidermal growth factor receptor
2,ERLOTINIB HYDROCHLORIDE,CHEMBL1079742,INHIBITOR,Epidermal growth factor receptor erbB1 inhibitor,Research / Unknown,4.0,Epidermal growth factor receptor
3,GEFITINIB,CHEMBL939,INHIBITOR,Epidermal growth factor receptor erbB1 inhibitor,Research / Unknown,4.0,Epidermal growth factor receptor
4,LAPATINIB DITOSYLATE,CHEMBL1201179,INHIBITOR,Epidermal growth factor receptor erbB1 inhibitor,Research / Unknown,4.0,Epidermal growth factor receptor
5,AFATINIB DIMALEATE,CHEMBL2105712,INHIBITOR,Epidermal growth factor receptor erbB1 inhibitor,Research / Unknown,4.0,Epidermal growth factor receptor
6,OSIMERTINIB MESYLATE,CHEMBL3545063,INHIBITOR,Epidermal growth factor receptor erbB1 inhibitor,Research / Unknown,4.0,Epidermal growth factor receptor
7,NECITUMUMAB,CHEMBL1743047,INHIBITOR,Epidermal growth factor receptor erbB1 inhibitor,Research / Unknown,4.0,Epidermal growth factor receptor
8,OLMUTINIB,CHEMBL3786343,INHIBITOR,Epidermal growth factor receptor erbB1 inhibitor,Research / Unknown,4.0,Epidermal growth factor receptor
9,BRIGATINIB,CHEMBL3545311,INHIBITOR,Epidermal growth factor receptor erbB1 inhibitor,Research / Unknown,4.0,Epidermal growth factor receptor


### 22. Save the recommendations dataset

In [22]:
recommendations_file = PROCESSED_DIR / f"{target_name.lower()}_drug_recommendations.csv"
drug_recommendations_df.to_csv(recommendations_file, index=False)
print("Saved:", recommendations_file)

mechanisms_file = PROCESSED_DIR / f"{target_name.lower()}_chembl_mechanisms.csv"
clean_mechanisms_df.to_csv(mechanisms_file, index=False)
print("Saved:", mechanisms_file)

Saved: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/processed/egfr_drug_recommendations.csv
Saved: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/processed/egfr_chembl_mechanisms.csv


### 23. Stage 2 summary

In [23]:
approved = drug_recommendations_df[drug_recommendations_df["approval_status"] == "Approved"]
print(f"Target: {target_name} ({target_chembl_id})")
print("=" * 60)
print(f"Total drugs with mechanism + name: {len(drug_recommendations_df)}")
print(f"Approved EGFR drugs: {len(approved)}")
print("\nApproved EGFR drugs:")
for _, r in approved.iterrows():
    print(f"  - {r['drug_name']}: {r['action_type']}")
drug_recommendations_df.head(25)

Target: EGFR (CHEMBL203)
Total drugs with mechanism + name: 76
Approved EGFR drugs: 0

Approved EGFR drugs:


,drug_name,molecule_chembl_id,action_type,mechanism_of_action,approval_status,max_phase,target_name
0,PANITUMUMAB,CHEMBL1201827,INHIBITOR,Epidermal growth factor receptor erbB1 inhibitor,Research / Unknown,4.0,Epidermal growth factor receptor
1,CETUXIMAB,CHEMBL1201577,INHIBITOR,Epidermal growth factor receptor erbB1 inhibitor,Research / Unknown,4.0,Epidermal growth factor receptor
2,ERLOTINIB HYDROCHLORIDE,CHEMBL1079742,INHIBITOR,Epidermal growth factor receptor erbB1 inhibitor,Research / Unknown,4.0,Epidermal growth factor receptor
3,GEFITINIB,CHEMBL939,INHIBITOR,Epidermal growth factor receptor erbB1 inhibitor,Research / Unknown,4.0,Epidermal growth factor receptor
4,LAPATINIB DITOSYLATE,CHEMBL1201179,INHIBITOR,Epidermal growth factor receptor erbB1 inhibitor,Research / Unknown,4.0,Epidermal growth factor receptor
5,AFATINIB DIMALEATE,CHEMBL2105712,INHIBITOR,Epidermal growth factor receptor erbB1 inhibitor,Research / Unknown,4.0,Epidermal growth factor receptor
6,OSIMERTINIB MESYLATE,CHEMBL3545063,INHIBITOR,Epidermal growth factor receptor erbB1 inhibitor,Research / Unknown,4.0,Epidermal growth factor receptor
7,NECITUMUMAB,CHEMBL1743047,INHIBITOR,Epidermal growth factor receptor erbB1 inhibitor,Research / Unknown,4.0,Epidermal growth factor receptor
8,OLMUTINIB,CHEMBL3786343,INHIBITOR,Epidermal growth factor receptor erbB1 inhibitor,Research / Unknown,4.0,Epidermal growth factor receptor
9,BRIGATINIB,CHEMBL3545311,INHIBITOR,Epidermal growth factor receptor erbB1 inhibitor,Research / Unknown,4.0,Epidermal growth factor receptor


# 01 — Data Exploration: EGFR Drug–Target Evidence (ChEMBL)

First dataset for the Therapeutic Strategy Assistant.
Flow: **EGFR → ChEMBL target → activity records → drug-target dataset → CSV**.

*Resilient:* retries on ChEMBL hiccups, and falls back to the known EGFR id `CHEMBL203` if the search fails.

### 1. Test notebook environment

In [ ]:
import sys
import time
from pathlib import Path

import requests
import pandas as pd

print("Notebook is working")
print("Python executable:", sys.executable)

Notebook is working
Python executable: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/.venv/bin/python


### 2. Helper: GET with retries
ChEMBL sometimes returns a temporary 500 or times out. This helper retries a few times so a hiccup does not crash the notebook.

In [ ]:
def chembl_get(url, params, retries=4, pause=3):
    """GET JSON from ChEMBL, retrying on 500 / timeout. Returns dict or None."""
    for attempt in range(retries):
        try:
            response = requests.get(url, params=params, timeout=(10, 120))
            if response.status_code == 200:
                return response.json()
            print(f"  attempt {attempt + 1}: HTTP {response.status_code}, retrying...")
        except requests.exceptions.RequestException as error:
            print(f"  attempt {attempt + 1}: {type(error).__name__}, retrying...")
        time.sleep(pause)
    return None

### 3. Set project folders

In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Processed data folder:", PROCESSED_DIR)

Project root: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant
Processed data folder: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/processed


### 4. Choose target

In [ ]:
target_name = "EGFR"

print("Target selected:", target_name)

Target selected: EGFR


### 5. Search ChEMBL for the target (with fallback)

In [ ]:
chembl_target_search_url = "https://www.ebi.ac.uk/chembl/api/data/target/search.json"

target_search_data = chembl_get(chembl_target_search_url, {"q": target_name})

if target_search_data is not None:
    print("ChEMBL target search worked.")
    print(target_search_data.keys())
else:
    print("ChEMBL target search failed after retries -> will use fallback CHEMBL203.")

ChEMBL target search worked.
dict_keys(['page_meta', 'targets'])


### 6. Convert target results to dataframe

In [ ]:
if target_search_data is not None:
    targets_df = pd.DataFrame(target_search_data.get("targets", []))
    print("Number of targets found:", len(targets_df))
else:
    targets_df = pd.DataFrame()

useful_target_columns = ["target_chembl_id", "pref_name", "organism", "target_type"]
available_target_columns = [c for c in useful_target_columns if c in targets_df.columns] or useful_target_columns

if not targets_df.empty:
    display(targets_df[available_target_columns].head(20))

Number of targets found: 20


,target_chembl_id,pref_name,organism,target_type
0,CHEMBL4523747,EGFR/PPP1CA,Homo sapiens,PROTEIN-PROTEIN INTERACTION
1,CHEMBL5465557,CCN2-EGFR,Homo sapiens,PROTEIN-PROTEIN INTERACTION
2,CHEMBL3608,Epidermal growth factor receptor,Mus musculus,SINGLE PROTEIN
3,CHEMBL6193842,Protein cereblon/Epidermal growth factor receptor,Mus musculus,PROTEIN-PROTEIN INTERACTION
4,CHEMBL203,Epidermal growth factor receptor,Homo sapiens,SINGLE PROTEIN
5,CHEMBL4523680,Protein cereblon/Epidermal growth factor receptor,Homo sapiens,PROTEIN-PROTEIN INTERACTION
6,CHEMBL2363049,Epidermal growth factor receptor,Homo sapiens,PROTEIN FAMILY
7,CHEMBL3137284,MER intracellular domain/EGFR extracellular do...,Homo sapiens,CHIMERIC PROTEIN
8,CHEMBL4523998,von Hippel-Lindau disease tumor suppressor/Epi...,Homo sapiens,PROTEIN-PROTEIN INTERACTION
9,CHEMBL6193841,Protein cereblon/Epidermal growth factor receptor,Mus musculus,PROTEIN-PROTEIN INTERACTION


### 7. Keep human single-protein targets

In [ ]:
if not targets_df.empty:
    human_targets = targets_df[
        (targets_df["organism"].str.contains("Homo sapiens", case=False, na=False))
        & (targets_df["target_type"] == "SINGLE PROTEIN")
    ].copy()
    display(human_targets[available_target_columns].head(10))
else:
    human_targets = pd.DataFrame()
    print("No targets dataframe; will use fallback.")

,target_chembl_id,pref_name,organism,target_type
4,CHEMBL203,Epidermal growth factor receptor,Homo sapiens,SINGLE PROTEIN
10,CHEMBL1824,Receptor tyrosine-protein kinase erbB-2,Homo sapiens,SINGLE PROTEIN
12,CHEMBL3009,Receptor tyrosine-protein kinase erbB-4,Homo sapiens,SINGLE PROTEIN
13,CHEMBL5838,Receptor tyrosine-protein kinase erbB-3,Homo sapiens,SINGLE PROTEIN


### 8. Pick the target id (prefer CHEMBL203)

In [ ]:
if not human_targets.empty:
    exact = human_targets[human_targets["target_chembl_id"] == "CHEMBL203"]
    selected_target = exact.iloc[0] if not exact.empty else human_targets.iloc[0]
    target_chembl_id = selected_target["target_chembl_id"]
    target_pref_name = selected_target["pref_name"]
else:
    target_chembl_id = "CHEMBL203"
    target_pref_name = "Epidermal growth factor receptor"

print("Selected target ChEMBL ID:", target_chembl_id)
print("Selected target name:", target_pref_name)

Selected target ChEMBL ID: CHEMBL203
Selected target name: Epidermal growth factor receptor


### 9. Get ChEMBL activity records
We filter to records that have a `pchembl_value` (a potency score). This keeps the response light so ChEMBL does not return a 500, and gives us better-quality data.

In [ ]:
chembl_activity_url = "https://www.ebi.ac.uk/chembl/api/data/activity.json"

activity_params = {
    "target_chembl_id": target_chembl_id,
    "pchembl_value__isnull": "false",
    "limit": 20,
}

activity_data = chembl_get(chembl_activity_url, activity_params)

if activity_data is None:
    raise RuntimeError("ChEMBL activity endpoint failed after retries. Try again shortly.")

print("Activity search worked.")
print(activity_data.keys())

Activity search worked.
dict_keys(['activities', 'page_meta'])


### 10. Convert activity records to dataframe

In [ ]:
activities = activity_data.get("activities", [])
activities_df = pd.DataFrame(activities)
print("Number of activity records found:", len(activities_df))
activities_df.head()

Number of activity records found: 20


,action_type,activity_comment,activity_id,activity_properties,assay_chembl_id,assay_description,assay_type,assay_variant_accession,assay_variant_mutation,bao_endpoint,...,target_organism,target_pref_name,target_tax_id,text_value,toid,type,units,uo_units,upper_value,value
0,None,None,32260,[],CHEMBL674637,Inhibitory activity towards tyrosine phosphory...,B,None,None,BAO_0000190,...,Homo sapiens,Epidermal growth factor receptor,9606,None,None,IC50,uM,UO_0000065,None,0.041
1,None,None,32263,[],CHEMBL621151,Inhibition of autophosphorylation of human epi...,F,None,None,BAO_0000190,...,Homo sapiens,Epidermal growth factor receptor,9606,None,None,IC50,uM,UO_0000065,None,0.3
2,None,None,32265,[],CHEMBL615325,Inhibition of ligand-induced proliferation in ...,F,None,None,BAO_0000190,...,Homo sapiens,Epidermal growth factor receptor,9606,None,None,IC50,uM,UO_0000065,None,7.82
3,None,None,32267,[],CHEMBL674637,Inhibitory activity towards tyrosine phosphory...,B,None,None,BAO_0000190,...,Homo sapiens,Epidermal growth factor receptor,9606,None,None,IC50,uM,UO_0000065,None,0.17
4,None,None,32270,[],CHEMBL621151,Inhibition of autophosphorylation of human epi...,F,None,None,BAO_0000190,...,Homo sapiens,Epidermal growth factor receptor,9606,None,None,IC50,uM,UO_0000065,None,0.04


### 11. Keep useful activity columns

In [ ]:
useful_activity_columns = [
    "molecule_chembl_id", "target_chembl_id", "target_organism", "target_pref_name",
    "assay_chembl_id", "assay_description", "standard_type", "standard_value",
    "standard_units", "pchembl_value", "document_chembl_id",
]

available_activity_columns = [c for c in useful_activity_columns if c in activities_df.columns]
clean_activities_df = activities_df[available_activity_columns].copy()
print("Rows:", len(clean_activities_df))
clean_activities_df.head(20)

Rows: 20


,molecule_chembl_id,target_chembl_id,target_organism,target_pref_name,assay_chembl_id,assay_description,standard_type,standard_value,standard_units,pchembl_value,document_chembl_id
0,CHEMBL68920,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL674637,Inhibitory activity towards tyrosine phosphory...,IC50,41.0,nM,7.39,CHEMBL1134862
1,CHEMBL68920,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL621151,Inhibition of autophosphorylation of human epi...,IC50,300.0,nM,6.52,CHEMBL1134862
2,CHEMBL68920,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL615325,Inhibition of ligand-induced proliferation in ...,IC50,7820.0,nM,5.11,CHEMBL1134862
3,CHEMBL69960,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL674637,Inhibitory activity towards tyrosine phosphory...,IC50,170.0,nM,6.77,CHEMBL1134862
4,CHEMBL69960,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL621151,Inhibition of autophosphorylation of human epi...,IC50,40.0,nM,7.40,CHEMBL1134862
5,CHEMBL69960,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL615325,Inhibition of ligand-induced proliferation in ...,IC50,440.0,nM,6.36,CHEMBL1134862
6,CHEMBL137635,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL677833,In vitro inhibition of Epidermal growth factor...,IC50,9300.0,nM,5.03,CHEMBL1145114
7,CHEMBL77085,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL674643,Inhibitory concentration of EGF dependent auto...,IC50,96000.0,nM,4.02,CHEMBL1124610
8,CHEMBL77085,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL675636,Inhibitory concentration of EGF dependent auto...,Ki,24000.0,nM,4.62,CHEMBL1124610
9,CHEMBL443268,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL674637,Inhibitory activity towards tyrosine phosphory...,IC50,5310.0,nM,5.28,CHEMBL1134862


### 12. Build the drug-target dataset

In [ ]:
clean_activities_df = clean_activities_df.dropna(subset=["molecule_chembl_id"]).copy()

drug_target_activity_df = clean_activities_df.drop_duplicates(
    subset=["molecule_chembl_id", "target_chembl_id"]
).copy()

drug_target_activity_df["target_name"] = target_pref_name
drug_target_activity_df["relationship_source"] = "ChEMBL activity"

print("Unique molecule-target relationships:", len(drug_target_activity_df))
drug_target_activity_df.head(20)

Unique molecule-target relationships: 12


,molecule_chembl_id,target_chembl_id,target_organism,target_pref_name,assay_chembl_id,assay_description,standard_type,standard_value,standard_units,pchembl_value,document_chembl_id,target_name,relationship_source
0,CHEMBL68920,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL674637,Inhibitory activity towards tyrosine phosphory...,IC50,41.0,nM,7.39,CHEMBL1134862,Epidermal growth factor receptor,ChEMBL activity
3,CHEMBL69960,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL674637,Inhibitory activity towards tyrosine phosphory...,IC50,170.0,nM,6.77,CHEMBL1134862,Epidermal growth factor receptor,ChEMBL activity
6,CHEMBL137635,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL677833,In vitro inhibition of Epidermal growth factor...,IC50,9300.0,nM,5.03,CHEMBL1145114,Epidermal growth factor receptor,ChEMBL activity
7,CHEMBL77085,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL674643,Inhibitory concentration of EGF dependent auto...,IC50,96000.0,nM,4.02,CHEMBL1124610,Epidermal growth factor receptor,ChEMBL activity
9,CHEMBL443268,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL674637,Inhibitory activity towards tyrosine phosphory...,IC50,5310.0,nM,5.28,CHEMBL1134862,Epidermal growth factor receptor,ChEMBL activity
10,CHEMBL76589,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL674643,Inhibitory concentration of EGF dependent auto...,IC50,125.0,nM,6.90,CHEMBL1124610,Epidermal growth factor receptor,ChEMBL activity
11,CHEMBL76904,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL674643,Inhibitory concentration of EGF dependent auto...,IC50,35000.0,nM,4.46,CHEMBL1124610,Epidermal growth factor receptor,ChEMBL activity
13,CHEMBL304271,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL674637,Inhibitory activity towards tyrosine phosphory...,IC50,0.45,nM,9.35,CHEMBL1134862,Epidermal growth factor receptor,ChEMBL activity
14,CHEMBL296407,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL674643,Inhibitory concentration of EGF dependent auto...,IC50,10000.0,nM,5.00,CHEMBL1124610,Epidermal growth factor receptor,ChEMBL activity
16,CHEMBL309625,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL674643,Inhibitory concentration of EGF dependent auto...,IC50,60000.0,nM,4.22,CHEMBL1124610,Epidermal growth factor receptor,ChEMBL activity


### 13. Add a simple confidence score

In [ ]:
def calculate_activity_confidence(row):
    score = 0.4
    if pd.notna(row.get("pchembl_value")):
        score += 0.25
    if pd.notna(row.get("standard_value")):
        score += 0.15
    if pd.notna(row.get("assay_description")):
        score += 0.1
    if pd.notna(row.get("document_chembl_id")):
        score += 0.1
    return round(min(score, 1.0), 2)


drug_target_activity_df["confidence_score"] = drug_target_activity_df.apply(
    calculate_activity_confidence, axis=1
)
drug_target_activity_df.head(20)

,molecule_chembl_id,target_chembl_id,target_organism,target_pref_name,assay_chembl_id,assay_description,standard_type,standard_value,standard_units,pchembl_value,document_chembl_id,target_name,relationship_source,confidence_score
0,CHEMBL68920,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL674637,Inhibitory activity towards tyrosine phosphory...,IC50,41.0,nM,7.39,CHEMBL1134862,Epidermal growth factor receptor,ChEMBL activity,1.0
3,CHEMBL69960,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL674637,Inhibitory activity towards tyrosine phosphory...,IC50,170.0,nM,6.77,CHEMBL1134862,Epidermal growth factor receptor,ChEMBL activity,1.0
6,CHEMBL137635,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL677833,In vitro inhibition of Epidermal growth factor...,IC50,9300.0,nM,5.03,CHEMBL1145114,Epidermal growth factor receptor,ChEMBL activity,1.0
7,CHEMBL77085,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL674643,Inhibitory concentration of EGF dependent auto...,IC50,96000.0,nM,4.02,CHEMBL1124610,Epidermal growth factor receptor,ChEMBL activity,1.0
9,CHEMBL443268,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL674637,Inhibitory activity towards tyrosine phosphory...,IC50,5310.0,nM,5.28,CHEMBL1134862,Epidermal growth factor receptor,ChEMBL activity,1.0
10,CHEMBL76589,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL674643,Inhibitory concentration of EGF dependent auto...,IC50,125.0,nM,6.90,CHEMBL1124610,Epidermal growth factor receptor,ChEMBL activity,1.0
11,CHEMBL76904,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL674643,Inhibitory concentration of EGF dependent auto...,IC50,35000.0,nM,4.46,CHEMBL1124610,Epidermal growth factor receptor,ChEMBL activity,1.0
13,CHEMBL304271,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL674637,Inhibitory activity towards tyrosine phosphory...,IC50,0.45,nM,9.35,CHEMBL1134862,Epidermal growth factor receptor,ChEMBL activity,1.0
14,CHEMBL296407,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL674643,Inhibitory concentration of EGF dependent auto...,IC50,10000.0,nM,5.00,CHEMBL1124610,Epidermal growth factor receptor,ChEMBL activity,1.0
16,CHEMBL309625,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL674643,Inhibitory concentration of EGF dependent auto...,IC50,60000.0,nM,4.22,CHEMBL1124610,Epidermal growth factor receptor,ChEMBL activity,1.0


### 14. Save the dataset

In [ ]:
activity_file = PROCESSED_DIR / f"{target_name.lower()}_chembl_activity.csv"
drug_target_activity_df.to_csv(activity_file, index=False)
print("Saved:", activity_file)

Saved: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/processed/egfr_chembl_activity.csv


### 15. Check saved files

In [ ]:
for file in PROCESSED_DIR.glob("*.csv"):
    print(file.name)

egfr_drug_recommendations.csv
egfr_chembl_mechanisms.csv
egfr_chembl_activity.csv


### 16. Final result

In [ ]:
print(f"Therapeutic Strategy Assistant Dataset for Target: {target_name}")
print("=" * 70)
print(f"Selected target ID: {target_chembl_id}")
print(f"Selected target name: {target_pref_name}")
print(f"Activity records: {len(clean_activities_df)}")
print(f"Unique molecule-target relationships: {len(drug_target_activity_df)}")
drug_target_activity_df.head(20)

Therapeutic Strategy Assistant Dataset for Target: EGFR
Selected target ID: CHEMBL203
Selected target name: Epidermal growth factor receptor
Activity records: 20
Unique molecule-target relationships: 12


,molecule_chembl_id,target_chembl_id,target_organism,target_pref_name,assay_chembl_id,assay_description,standard_type,standard_value,standard_units,pchembl_value,document_chembl_id,target_name,relationship_source,confidence_score
0,CHEMBL68920,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL674637,Inhibitory activity towards tyrosine phosphory...,IC50,41.0,nM,7.39,CHEMBL1134862,Epidermal growth factor receptor,ChEMBL activity,1.0
3,CHEMBL69960,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL674637,Inhibitory activity towards tyrosine phosphory...,IC50,170.0,nM,6.77,CHEMBL1134862,Epidermal growth factor receptor,ChEMBL activity,1.0
6,CHEMBL137635,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL677833,In vitro inhibition of Epidermal growth factor...,IC50,9300.0,nM,5.03,CHEMBL1145114,Epidermal growth factor receptor,ChEMBL activity,1.0
7,CHEMBL77085,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL674643,Inhibitory concentration of EGF dependent auto...,IC50,96000.0,nM,4.02,CHEMBL1124610,Epidermal growth factor receptor,ChEMBL activity,1.0
9,CHEMBL443268,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL674637,Inhibitory activity towards tyrosine phosphory...,IC50,5310.0,nM,5.28,CHEMBL1134862,Epidermal growth factor receptor,ChEMBL activity,1.0
10,CHEMBL76589,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL674643,Inhibitory concentration of EGF dependent auto...,IC50,125.0,nM,6.90,CHEMBL1124610,Epidermal growth factor receptor,ChEMBL activity,1.0
11,CHEMBL76904,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL674643,Inhibitory concentration of EGF dependent auto...,IC50,35000.0,nM,4.46,CHEMBL1124610,Epidermal growth factor receptor,ChEMBL activity,1.0
13,CHEMBL304271,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL674637,Inhibitory activity towards tyrosine phosphory...,IC50,0.45,nM,9.35,CHEMBL1134862,Epidermal growth factor receptor,ChEMBL activity,1.0
14,CHEMBL296407,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL674643,Inhibitory concentration of EGF dependent auto...,IC50,10000.0,nM,5.00,CHEMBL1124610,Epidermal growth factor receptor,ChEMBL activity,1.0
16,CHEMBL309625,CHEMBL203,Homo sapiens,Epidermal growth factor receptor,CHEMBL674643,Inhibitory concentration of EGF dependent auto...,IC50,60000.0,nM,4.22,CHEMBL1124610,Epidermal growth factor receptor,ChEMBL activity,1.0


### 17. Get EGFR drug mechanisms

In [ ]:
chembl_mechanism_url = "https://www.ebi.ac.uk/chembl/api/data/mechanism.json"

mechanism_data = chembl_get(chembl_mechanism_url, {"target_chembl_id": target_chembl_id, "limit": 100})

if mechanism_data is None:
    raise RuntimeError("ChEMBL mechanism endpoint failed after retries. Try again shortly.")

mechanisms = mechanism_data.get("mechanisms", [])
mechanisms_df = pd.DataFrame(mechanisms)
print("Mechanism records found:", len(mechanisms_df))
mechanisms_df.head()

Mechanism records found: 90


,action_type,binding_site_comment,direct_interaction,disease_efficacy,max_phase,mec_id,mechanism_comment,mechanism_of_action,mechanism_refs,molecular_mechanism,molecule_chembl_id,parent_molecule_chembl_id,record_id,selectivity_comment,site_id,target_chembl_id,variant_sequence
0,INHIBITOR,None,1,1,4,241,NaN,Epidermal growth factor receptor erbB1 inhibitor,[{'ref_id': 'setid=e0fa4bca-f245-4d92-ae29-b0c...,1,CHEMBL1201827,CHEMBL1201827,1390843,NaN,None,CHEMBL203,None
1,INHIBITOR,None,1,1,4,242,NaN,Epidermal growth factor receptor erbB1 inhibitor,[{'ref_id': 'setid=8bc6397e-4bd8-4d37-a007-a32...,1,CHEMBL1201577,CHEMBL1201577,1344036,NaN,None,CHEMBL203,None
2,INHIBITOR,None,1,1,4,243,NaN,Epidermal growth factor receptor erbB1 inhibitor,[{'ref_id': 'setid=57bccb29-1c47-4c64-ab6a-779...,1,CHEMBL1079742,CHEMBL553,1344174,NaN,None,CHEMBL203,None
3,INHIBITOR,None,1,1,4,244,NaN,Epidermal growth factor receptor erbB1 inhibitor,[{'ref_id': 'setid=827d60e8-7e07-41b7-c28b-49e...,1,CHEMBL939,CHEMBL939,1344994,NaN,None,CHEMBL203,None
4,INHIBITOR,None,1,1,4,245,NaN,Epidermal growth factor receptor erbB1 inhibitor,[{'ref_id': 'setid=63319b01-cad6-4d0a-c39b-938...,1,CHEMBL1201179,CHEMBL554,1343218,NaN,None,CHEMBL203,None


### 18. Keep useful mechanism columns

In [ ]:
useful_mechanism_columns = [
    "molecule_chembl_id", "mechanism_of_action", "action_type",
    "target_chembl_id", "mechanism_refs",
]
available_mechanism_columns = [c for c in useful_mechanism_columns if c in mechanisms_df.columns]
clean_mechanisms_df = mechanisms_df[available_mechanism_columns].drop_duplicates(subset=["molecule_chembl_id"]).copy()
print("Unique drugs with a mechanism:", len(clean_mechanisms_df))
clean_mechanisms_df.head(20)

Unique drugs with a mechanism: 76


,molecule_chembl_id,mechanism_of_action,action_type,target_chembl_id,mechanism_refs
0,CHEMBL1201827,Epidermal growth factor receptor erbB1 inhibitor,INHIBITOR,CHEMBL203,[{'ref_id': 'setid=e0fa4bca-f245-4d92-ae29-b0c...
1,CHEMBL1201577,Epidermal growth factor receptor erbB1 inhibitor,INHIBITOR,CHEMBL203,[{'ref_id': 'setid=8bc6397e-4bd8-4d37-a007-a32...
2,CHEMBL1079742,Epidermal growth factor receptor erbB1 inhibitor,INHIBITOR,CHEMBL203,[{'ref_id': 'setid=57bccb29-1c47-4c64-ab6a-779...
3,CHEMBL939,Epidermal growth factor receptor erbB1 inhibitor,INHIBITOR,CHEMBL203,[{'ref_id': 'setid=827d60e8-7e07-41b7-c28b-49e...
4,CHEMBL1201179,Epidermal growth factor receptor erbB1 inhibitor,INHIBITOR,CHEMBL203,[{'ref_id': 'setid=63319b01-cad6-4d0a-c39b-938...
5,CHEMBL2105712,Epidermal growth factor receptor erbB1 inhibitor,INHIBITOR,CHEMBL203,[{'ref_id': 'fd638e5e-8032-e7ca-0179-95e96ab5d...
6,CHEMBL3545063,Epidermal growth factor receptor erbB1 inhibitor,INHIBITOR,CHEMBL203,[{'ref_id': 'setid=5e81b4a7-b971-45e1-9c31-29c...
7,CHEMBL1743047,Epidermal growth factor receptor erbB1 inhibitor,INHIBITOR,CHEMBL203,[{'ref_id': 'setid=89bcf553-669a-40b0-a9d7-67a...
8,CHEMBL1645462,Epidermal growth factor receptor erbB1 inhibitor,INHIBITOR,CHEMBL203,"[{'ref_id': '17062696', 'ref_type': 'PubMed', ..."
9,CHEMBL1947204,Epidermal growth factor receptor erbB1 inhibitor,INHIBITOR,CHEMBL203,"[{'ref_id': '21789172', 'ref_type': 'PubMed', ..."


### 19. Helper: look up a drug name + approval phase
Each `molecule_chembl_id` (e.g. CHEMBL939) is turned into a readable name (e.g. GEFITINIB) via ChEMBL's molecule endpoint. `max_phase = 4` means the drug is **approved**.

In [ ]:
def get_molecule_info(chembl_id):
    """Return (pref_name, max_phase) for a molecule id, or (None, None)."""
    url = f"https://www.ebi.ac.uk/chembl/api/data/molecule/{chembl_id}.json"
    data = chembl_get(url, params=None)
    if data is None:
        return None, None
    return data.get("pref_name"), data.get("max_phase")

### 20. Enrich each drug with its name (this calls the API per drug, so it takes ~1 min)

In [ ]:
names, phases = [], []
for i, cid in enumerate(clean_mechanisms_df["molecule_chembl_id"], start=1):
    name, phase = get_molecule_info(cid)
    names.append(name)
    phases.append(phase)
    if i % 10 == 0:
        print(f"  enriched {i}/{len(clean_mechanisms_df)}")

clean_mechanisms_df["drug_name"] = names
clean_mechanisms_df["max_phase"] = phases
print("Done. Drugs with a real name:", clean_mechanisms_df["drug_name"].notna().sum())
clean_mechanisms_df.head(20)

  enriched 10/76
  enriched 20/76
  enriched 30/76
  enriched 40/76
  enriched 50/76
  enriched 60/76
  enriched 70/76
Done. Drugs with a real name: 76


,molecule_chembl_id,mechanism_of_action,action_type,target_chembl_id,mechanism_refs,drug_name,max_phase
0,CHEMBL1201827,Epidermal growth factor receptor erbB1 inhibitor,INHIBITOR,CHEMBL203,[{'ref_id': 'setid=e0fa4bca-f245-4d92-ae29-b0c...,PANITUMUMAB,4.0
1,CHEMBL1201577,Epidermal growth factor receptor erbB1 inhibitor,INHIBITOR,CHEMBL203,[{'ref_id': 'setid=8bc6397e-4bd8-4d37-a007-a32...,CETUXIMAB,4.0
2,CHEMBL1079742,Epidermal growth factor receptor erbB1 inhibitor,INHIBITOR,CHEMBL203,[{'ref_id': 'setid=57bccb29-1c47-4c64-ab6a-779...,ERLOTINIB HYDROCHLORIDE,4.0
3,CHEMBL939,Epidermal growth factor receptor erbB1 inhibitor,INHIBITOR,CHEMBL203,[{'ref_id': 'setid=827d60e8-7e07-41b7-c28b-49e...,GEFITINIB,4.0
4,CHEMBL1201179,Epidermal growth factor receptor erbB1 inhibitor,INHIBITOR,CHEMBL203,[{'ref_id': 'setid=63319b01-cad6-4d0a-c39b-938...,LAPATINIB DITOSYLATE,4.0
5,CHEMBL2105712,Epidermal growth factor receptor erbB1 inhibitor,INHIBITOR,CHEMBL203,[{'ref_id': 'fd638e5e-8032-e7ca-0179-95e96ab5d...,AFATINIB DIMALEATE,4.0
6,CHEMBL3545063,Epidermal growth factor receptor erbB1 inhibitor,INHIBITOR,CHEMBL203,[{'ref_id': 'setid=5e81b4a7-b971-45e1-9c31-29c...,OSIMERTINIB MESYLATE,4.0
7,CHEMBL1743047,Epidermal growth factor receptor erbB1 inhibitor,INHIBITOR,CHEMBL203,[{'ref_id': 'setid=89bcf553-669a-40b0-a9d7-67a...,NECITUMUMAB,4.0
8,CHEMBL1645462,Epidermal growth factor receptor erbB1 inhibitor,INHIBITOR,CHEMBL203,"[{'ref_id': '17062696', 'ref_type': 'PubMed', ...",AC-480,1.0
9,CHEMBL1947204,Epidermal growth factor receptor erbB1 inhibitor,INHIBITOR,CHEMBL203,"[{'ref_id': '21789172', 'ref_type': 'PubMed', ...",ALLITINIB,2.0


### 21. Build the drug recommendations table

In [ ]:
drug_recommendations_df = clean_mechanisms_df[
    clean_mechanisms_df["drug_name"].notna()
].copy()

# approval label from max_phase
def approval_label(phase):
    if phase == 4:
        return "Approved"
    if phase in (1, 2, 3):
        return f"Investigational (Phase {int(phase)})"
    return "Research / Unknown"

drug_recommendations_df["approval_status"] = drug_recommendations_df["max_phase"].apply(approval_label)
drug_recommendations_df["target_name"] = target_pref_name

drug_recommendations_df = drug_recommendations_df[[
    "drug_name", "molecule_chembl_id", "action_type",
    "mechanism_of_action", "approval_status", "max_phase", "target_name",
]].sort_values("max_phase", ascending=False, na_position="last").reset_index(drop=True)

print("Drug recommendations:", len(drug_recommendations_df))
drug_recommendations_df.head(25)

Drug recommendations: 76


,drug_name,molecule_chembl_id,action_type,mechanism_of_action,approval_status,max_phase,target_name
0,PANITUMUMAB,CHEMBL1201827,INHIBITOR,Epidermal growth factor receptor erbB1 inhibitor,Research / Unknown,4.0,Epidermal growth factor receptor
1,CETUXIMAB,CHEMBL1201577,INHIBITOR,Epidermal growth factor receptor erbB1 inhibitor,Research / Unknown,4.0,Epidermal growth factor receptor
2,ERLOTINIB HYDROCHLORIDE,CHEMBL1079742,INHIBITOR,Epidermal growth factor receptor erbB1 inhibitor,Research / Unknown,4.0,Epidermal growth factor receptor
3,GEFITINIB,CHEMBL939,INHIBITOR,Epidermal growth factor receptor erbB1 inhibitor,Research / Unknown,4.0,Epidermal growth factor receptor
4,LAPATINIB DITOSYLATE,CHEMBL1201179,INHIBITOR,Epidermal growth factor receptor erbB1 inhibitor,Research / Unknown,4.0,Epidermal growth factor receptor
5,AFATINIB DIMALEATE,CHEMBL2105712,INHIBITOR,Epidermal growth factor receptor erbB1 inhibitor,Research / Unknown,4.0,Epidermal growth factor receptor
6,OSIMERTINIB MESYLATE,CHEMBL3545063,INHIBITOR,Epidermal growth factor receptor erbB1 inhibitor,Research / Unknown,4.0,Epidermal growth factor receptor
7,NECITUMUMAB,CHEMBL1743047,INHIBITOR,Epidermal growth factor receptor erbB1 inhibitor,Research / Unknown,4.0,Epidermal growth factor receptor
8,OLMUTINIB,CHEMBL3786343,INHIBITOR,Epidermal growth factor receptor erbB1 inhibitor,Research / Unknown,4.0,Epidermal growth factor receptor
9,BRIGATINIB,CHEMBL3545311,INHIBITOR,Epidermal growth factor receptor erbB1 inhibitor,Research / Unknown,4.0,Epidermal growth factor receptor


### 22. Save the recommendations dataset

In [ ]:
recommendations_file = PROCESSED_DIR / f"{target_name.lower()}_drug_recommendations.csv"
drug_recommendations_df.to_csv(recommendations_file, index=False)
print("Saved:", recommendations_file)

mechanisms_file = PROCESSED_DIR / f"{target_name.lower()}_chembl_mechanisms.csv"
clean_mechanisms_df.to_csv(mechanisms_file, index=False)
print("Saved:", mechanisms_file)

Saved: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/processed/egfr_drug_recommendations.csv
Saved: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/processed/egfr_chembl_mechanisms.csv


### 23. Stage 2 summary

In [ ]:
approved = drug_recommendations_df[drug_recommendations_df["approval_status"] == "Approved"]
print(f"Target: {target_name} ({target_chembl_id})")
print("=" * 60)
print(f"Total drugs with mechanism + name: {len(drug_recommendations_df)}")
print(f"Approved EGFR drugs: {len(approved)}")
print("\nApproved EGFR drugs:")
for _, r in approved.iterrows():
    print(f"  - {r['drug_name']}: {r['action_type']}")
drug_recommendations_df.head(25)

Target: EGFR (CHEMBL203)
Total drugs with mechanism + name: 76
Approved EGFR drugs: 0

Approved EGFR drugs:


,drug_name,molecule_chembl_id,action_type,mechanism_of_action,approval_status,max_phase,target_name
0,PANITUMUMAB,CHEMBL1201827,INHIBITOR,Epidermal growth factor receptor erbB1 inhibitor,Research / Unknown,4.0,Epidermal growth factor receptor
1,CETUXIMAB,CHEMBL1201577,INHIBITOR,Epidermal growth factor receptor erbB1 inhibitor,Research / Unknown,4.0,Epidermal growth factor receptor
2,ERLOTINIB HYDROCHLORIDE,CHEMBL1079742,INHIBITOR,Epidermal growth factor receptor erbB1 inhibitor,Research / Unknown,4.0,Epidermal growth factor receptor
3,GEFITINIB,CHEMBL939,INHIBITOR,Epidermal growth factor receptor erbB1 inhibitor,Research / Unknown,4.0,Epidermal growth factor receptor
4,LAPATINIB DITOSYLATE,CHEMBL1201179,INHIBITOR,Epidermal growth factor receptor erbB1 inhibitor,Research / Unknown,4.0,Epidermal growth factor receptor
5,AFATINIB DIMALEATE,CHEMBL2105712,INHIBITOR,Epidermal growth factor receptor erbB1 inhibitor,Research / Unknown,4.0,Epidermal growth factor receptor
6,OSIMERTINIB MESYLATE,CHEMBL3545063,INHIBITOR,Epidermal growth factor receptor erbB1 inhibitor,Research / Unknown,4.0,Epidermal growth factor receptor
7,NECITUMUMAB,CHEMBL1743047,INHIBITOR,Epidermal growth factor receptor erbB1 inhibitor,Research / Unknown,4.0,Epidermal growth factor receptor
8,OLMUTINIB,CHEMBL3786343,INHIBITOR,Epidermal growth factor receptor erbB1 inhibitor,Research / Unknown,4.0,Epidermal growth factor receptor
9,BRIGATINIB,CHEMBL3545311,INHIBITOR,Epidermal growth factor receptor erbB1 inhibitor,Research / Unknown,4.0,Epidermal growth factor receptor
